# General Radial-Spectrum Line Scattering

This notebook uses a sampled isotropic radial k spectrum, such as `gaussian_radial`, and computes the covariance numerically as an average of `sinc(k r)`. It does not use the monochromatic `sin(k0 r)/(k0 r)` covariance except as an optional reference.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import rw_line_scattering as ris

plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## Radial Spectrum

In [ ]:
k0_nominal = 1.0
k_distribution = "gaussian_radial"
num_modes_k = 2**10
r_sigma_k = 0.01
random_seed = 12345

# Options: "qmc", "random", or "quadrature".
k_sampling = "qmc"

if k_sampling == "quadrature":
    k_radii, k_weights = ris.make_radial_k_quadrature(
        num_modes_k,
        k_distribution,
        k0=k0_nominal,
        sigma_k=r_sigma_k*k0_nominal,
    )
else:
    k_rng = np.random.default_rng(random_seed)
    k_sets = ris.make_field_k_sets(
        num_modes_k,
        k_distribution,
        k_rng,
        k0=k0_nominal,
        r_sigma_k=r_sigma_k,
        shared_k_vectors=True,
        use_qmc_k=(k_sampling == "qmc"),
        qmc_seed=random_seed,
    )
    k_radii = ris.k_radii_from_vectors(k_sets.psi1)
    k_weights = None

a = ris.gradient_variance_from_k_radii(k_radii, k_weights=k_weights)
rho0 = ris.rho0_from_k_radii(k_radii, k_weights=k_weights)
k_eff = np.sqrt(3*a)

k_mean = np.mean(k_radii) if k_weights is None else np.dot(k_weights, k_radii)
k_std = np.std(k_radii) if k_weights is None else np.sqrt(np.dot(k_weights, (k_radii-k_mean)**2))

print(f"distribution = {k_distribution}")
print(f"k sampling = {k_sampling}")
print(f"nominal k0 = {k0_nominal:.12g}")
print(f"effective k_eff = sqrt(<k^2>) = {k_eff:.12g}")
print(f"mean |k| = {k_mean:.12g}")
print(f"std |k| = {k_std:.12g}")
print(f"a = <k^2>/3 = {a:.12g}")
print(f"rho0 = a/pi = {rho0:.12g}")
print(f"rho0^2 = {rho0**2:.12g}")


In [ ]:
fig, ax = plt.subplots()
ax.hist(k_radii, bins=80, density=True, weights=k_weights, alpha=0.75)
ax.axvline(k0_nominal, color="tab:orange", linestyle="--", label="nominal k0")
ax.axvline(k_eff, color="tab:green", linestyle="--", label="sqrt(<k^2>)")
ax.set_xlabel("|k|")
ax.set_ylabel("density")
ax.legend()
plt.show()


## Grids

In [ ]:
r_min = 1e-3 / k_eff
r_max = 5e2 / k_eff
Nr = 10000
r_grid = np.linspace(r_min, r_max, Nr)
# r_grid = np.logspace(np.log10(r_min), np.log10(r_max), Nr)

Q_min = 0.1 * k_eff
Q_max = 20 * k_eff
NQ = 256
Q_grid = np.logspace(np.log10(Q_min), np.log10(Q_max), NQ)

tail_start = 0.8 * r_max

print(f"r range: {r_grid[0]:.6g} to {r_grid[-1]:.6g}; Nr={len(r_grid)}; dr={r_grid[1]-r_grid[0]:.6g}")
print(f"Q/k_eff range: {Q_grid[0]/k_eff:.6g} to {Q_grid[-1]/k_eff:.6g}; NQ={len(Q_grid)}")
print(f"tail window starts at r = {tail_start:.6g}")

## Numerical Covariance

In [ ]:
g_num, gp_num, gpp_num = ris.radial_covariance_numeric(r_grid, k_radii, k_weights=k_weights)

fig, ax = plt.subplots()
ax.plot(r_grid, g_num, label="g numeric")
ax.plot(r_grid, gp_num, label="g' numeric")
ax.plot(r_grid, gpp_num, label="g'' numeric")
ax.set_xlabel("r")
ax.set_xscale("log")
ax.legend()
plt.show()


## Line-Density Correlation

In [ ]:
N_samp = 2**16
use_qmc = True

M_J, C_L = ris.compute_CL_general(
    r_grid,
    k_radii,
    N_samp,
    k_weights=k_weights,
    use_qmc=use_qmc,
    random_seed=random_seed,
    progress=True,
)
C_L_coherent = ris.coherent_CL_general(C_L, k_radii, k_weights=k_weights)

print(f"empirical M_J(r_min={r_grid[0]:.6g}) = {M_J[0]:.12g}")
print(f"empirical M_J(r_max={r_grid[-1]:.6g}) = {M_J[-1]:.12g}")


In [ ]:
fig, ax = plt.subplots()
ax.plot(r_grid, M_J, label="M_J(r)")
ax.axhline(2*a*a, color="tab:orange", linestyle="--", label="2 a^2")
ax.axhline(4*a*a, color="tab:green", linestyle="--", label="4 a^2")
ax.set_xlabel("r")
ax.set_ylabel("M_J")
ax.set_xscale("log")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.loglog(r_grid, C_L, label="C_L")
ax.loglog(r_grid, rho0/(2*np.pi*r_grid**2), "--", label="rho0/(2 pi r^2)")
ax.axhline(rho0**2, color="tab:green", linestyle=":", label="rho0^2")
ax.set_xlabel("r")
ax.set_ylabel("C_L")
ax.set_ylim(1e-5, 1e1)
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.semilogx(r_grid, C_L_coherent, label="C_L(r) - rho0^2")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("r")
ax.set_ylabel("coherent C_L")
ax.legend()
plt.show()

## Scattering Transform

In [ ]:
diag = ris.compute_coherent_transform_diagnostics(
    r_grid,
    C_L,
    Q_grid,
    rho0,
    r_taper_start=tail_start,
)

W_tail = diag["w_tail"]
C_L_coherent = diag["CL_coherent"]
C_L_transform_raw = diag["CL_transform_raw"]
C_L_transform = diag["CL_transform"]

I_Q_raw = diag["I_coherent_raw"]
I_Q = diag["I_coherent_windowed"]
I_Q_full_direct = diag["I_full_windowed"]
I_plateau_windowed = diag["I_plateau_windowed"]

high_q = (Q_grid/k_eff >= 5.0) & (Q_grid/k_eff <= 30.0)
plateau_median = np.median(Q_grid[high_q] * I_Q[high_q])
print(f"tail window starts at r = {tail_start:.6g}")
print(f"CL[-1] = {C_L[-1]:.12g}")
print(f"CL[-1] - rho0^2 = {C_L[-1] - rho0**2:.12g}")
print(f"median Q*I(Q), Q/k_eff in [5, 30] = {plateau_median:.12g}")
print(f"expected local-line plateau pi*rho0 = {np.pi*rho0:.12g}")


In [ ]:
fig, ax = plt.subplots()
ax.loglog(Q_grid/k_eff, I_Q, ".-", linewidth=2.2, label="bg + window")
# ax.loglog(Q_grid/k_eff, I_Q_raw, alpha=0.55, label="coherent raw")
# ax.loglog(Q_grid/k_eff, I_Q_full_direct, alpha=0.35, label="full C_L windowed")
ax.loglog(Q_grid/k_eff, np.pi*rho0/Q_grid, "--", label="continuum")
ax.set_xlabel("Q/k_eff")
ax.set_ylabel("I(Q)")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.semilogx(Q_grid/k_eff, Q_grid * I_Q, linewidth=2.2, label="bg + window")
# ax.semilogx(Q_grid/k_eff, Q_grid * I_Q_raw, alpha=0.55, label="coherent raw")
ax.axhline(np.pi*rho0, color="tab:orange", linestyle="--", label="continuum")
ax.set_xlabel("Q/k_eff")
ax.set_ylabel("Q I(Q)")
ax.legend()
plt.show()

## Save Data

In [ ]:
output_dir = Path("rw_line_general_output")
output_dir.mkdir(exist_ok=True)

np.savez(
    output_dir / "general_data.npz",
    r_grid=r_grid,
    Q_grid=Q_grid,
    k_radii=k_radii,
    k_weights=k_weights,
    M_J=M_J,
    C_L=C_L,
    C_L_coherent=C_L_coherent,
    C_L_transform=C_L_transform,
    C_L_transform_raw=C_L_transform_raw,
    W_tail=W_tail,
    I_Q=I_Q,
    I_Q_raw=I_Q_raw,
    I_Q_full_direct=I_Q_full_direct,
    I_plateau_windowed=I_plateau_windowed,
    k0_nominal=k0_nominal,
    k_eff=k_eff,
    a=a,
    rho0=rho0,
    k_distribution=k_distribution,
    k_sampling=k_sampling,
    r_sigma_k=r_sigma_k,
    N_samp=N_samp,
    random_seed=random_seed,
)
print(f"saved data to {output_dir.resolve()}")
